In [1]:
%pip install Dash scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Hides notebook codeblocks so users can focus on the graphical plots

from IPython.display import HTML

HTML('''<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit" value="Click here to toggle on/off the raw code."></form>''')

In [3]:
import sklearn
print("scikit-learn version:", sklearn.__version__) 

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

print("StandardScaler and PCA imported successfully")

scikit-learn version: 1.8.0
StandardScaler and PCA imported successfully


In [4]:
# CELL 1 — Setup & imports
# (If you need packages: run pip install pandas numpy plotly ipywidgets scikit-learn first)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.graph_objs import FigureWidget
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Default locations and minimal element list (edit if your data lives elsewhere)
DATA_DIR = Path("data")
DEFAULT_ELEMENTS = ["Rb", "Sr", "Zr"]

## Loading tables from Google Drive
### Designed for the Google Sheet to have three tabs:
*  **srcs** tab contains source sample geochemistry and Group variable (follow example provided)
*  **srcs_locs** tab contains Latitude and Longitude coordinates for each source
*  **study** tab contains geochemistry of samples under study

In [5]:
# ============================================================
# CELL 2 — Configuration and loading helper
# ============================================================

from pathlib  import Path
from io       import StringIO
import pandas  as pd
import urllib.request

# ── Mode configuration ───────────────────────────────────────
USE_GOOGLE_SHEETS = True                           # ← default to Sheets

SHEET_ID = "1R4PlMACBn0l8ZguwtYDlZLnbvORzH5CHhKGFBezjSvk"           # ← paste your Sheet ID here

SHEET_TABS = {
    "srcs"      : "KRA21_sources",
    "srcs_locs" : "source_coords",
    "study"     : "study_samples",
}

CSV_FILES = {                                      # ← fallback filenames
    "srcs"      : "KRA21_sources.csv",
    "srcs_locs" : "source_coords.csv",
    "study"     : "study_samples.csv",
}

# ── Helpers ──────────────────────────────────────────────────

def gsheet_csv_url(sheet_id: str, sheet_name: str) -> str:
    """Build a public Google Sheets CSV-export URL."""
    base = "https://docs.google.com/spreadsheets/d"
    return f"{base}/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"


def _load_from_gsheets(sheet_id: str, sheet_name: str) -> pd.DataFrame:
    """
    Download a single tab from a public Google Sheet as a DataFrame.
    Uses urllib so there is no requests dependency.
    """
    url = gsheet_csv_url(sheet_id, sheet_name)
    print(f"  → fetching '{sheet_name}' from Google Sheets...")
    try:
        with urllib.request.urlopen(url, timeout=15) as resp:
            content = resp.read().decode("utf-8")
        df = pd.read_csv(StringIO(content))
        print(f"     loaded  {df.shape[0]} rows × {df.shape[1]} cols ✓")
        return df
    except urllib.error.HTTPError as e:
        raise RuntimeError(
            f"HTTP {e.code} fetching sheet '{sheet_name}'.\n"
            f"  URL : {url}\n"
            f"  Tip : make sure the Google Sheet is shared as\n"
            f"        'Anyone with the link → Viewer'."
        ) from e
    except urllib.error.URLError as e:
        raise RuntimeError(
            f"Network error fetching sheet '{sheet_name}'.\n"
            f"  Reason : {e.reason}\n"
            f"  Tip    : check your internet connection."
        ) from e


def _load_from_upload() -> pd.DataFrame:
    """
    Prompt the user to upload a CSV file.
    Works in Jupyter / JupyterLab / Binder.
    Falls back to a manual path prompt if ipywidgets is unavailable.
    """
    try:
        # ── ipywidgets upload widget (preferred) ─────────────
        import ipywidgets as widgets
        from IPython.display import display as ipy_display

        print("  → no local file found — please upload a CSV:")
        uploader = widgets.FileUpload(accept=".csv", multiple=False)
        ipy_display(uploader)

        # Poll until the user uploads something
        import time
        timeout, elapsed = 120, 0
        while not uploader.value and elapsed < timeout:
            time.sleep(1)
            elapsed += 1

        if not uploader.value:
            raise TimeoutError("No file uploaded within 120 seconds.")

        # ipywidgets ≥ 8  →  value is a dict keyed by filename
        uploaded = list(uploader.value.values())[0]
        content  = uploaded["content"]                 # bytes
        df       = pd.read_csv(StringIO(content.decode("utf-8")))
        print(f"     loaded  {df.shape[0]} rows × {df.shape[1]} cols ✓")
        return df

    except ImportError:
        # ── plain input() fallback (VS Code / terminal) ──────
        path_str = input(
            "  ipywidgets not available.\n"
            "  Enter the full path to your CSV file: "
        ).strip().strip('"').strip("'")

        path = Path(path_str)
        if not path.exists():
            raise FileNotFoundError(
                f"File not found: {path.resolve()}\n"
                f"  Tip : drag the file into the same folder as "
                f"this notebook and enter just the filename."
            )
        df = pd.read_csv(path)
        print(f"     loaded  {df.shape[0]} rows × {df.shape[1]} cols ✓")
        return df


def load_table(key: str) -> pd.DataFrame:
    """
    Load one project table by its key  ('srcs', 'srcs_locs', 'study').

    Priority
    --------
    1. Google Sheets  — if USE_GOOGLE_SHEETS = True
    2. Local CSV      — looks in current working directory
    3. Upload prompt  — if neither of the above works
    """
    if USE_GOOGLE_SHEETS:
        return _load_from_gsheets(SHEET_ID, SHEET_TABS[key])

    # ── local CSV attempt ─────────────────────────────────────
    csv_name = CSV_FILES[key]
    local    = Path.cwd() / csv_name

    if local.exists():
        print(f"  → reading local file: {csv_name}")
        try:
            df = pd.read_csv(local)
            print(f"     loaded  {df.shape[0]} rows × {df.shape[1]} cols ✓")
            return df
        except Exception as e:
            raise RuntimeError(f"Failed to read {csv_name}: {e}") from e

    # ── neither found → prompt ────────────────────────────────
    print(f"\n  ⚠ '{csv_name}' not found in {Path.cwd()}")
    print(f"  Falling back to upload prompt for '{key}'...\n")
    return _load_from_upload()


# ── Config summary ───────────────────────────────────────────
print("=" * 48)
print(" CELL 2 — Data loading configuration")
print("=" * 48)
print(f"  Mode      : {'Google Sheets ☁' if USE_GOOGLE_SHEETS else 'Local CSV / Upload'}")
if USE_GOOGLE_SHEETS:
    sid_display = SHEET_ID if SHEET_ID != "REPLACE_WITH_YOUR_SHEET_ID" \
                  else "⚠  NOT SET — paste your Sheet ID above"
    print(f"  Sheet ID  : {sid_display}")
print(f"  Tables    :")
for k in SHEET_TABS:
    src = SHEET_TABS[k] if USE_GOOGLE_SHEETS else CSV_FILES[k]
    print(f"    {k:<12} ←  {src}")
print("=" * 48)

 CELL 2 — Data loading configuration
  Mode      : Google Sheets ☁
  Sheet ID  : 1R4PlMACBn0l8ZguwtYDlZLnbvORzH5CHhKGFBezjSvk
  Tables    :
    srcs         ←  KRA21_sources
    srcs_locs    ←  source_coords
    study        ←  study_samples


### Dataset headers are cleaned up (if necessary) ###

In [6]:
# ============================================================
# CELL 3 — Cleaning & validation utilities
# ============================================================

import re
import pandas as pd

# ── Bruker XRF suffix patterns to strip ─────────────────────
# Covers common Bruker spectral line suffixes
BRUKER_SUFFIX_PATTERN = re.compile(
    r'\s*(K[ab][12]?|L[ab][12]?|M[ab][12]?)\s*$',
    flags=re.IGNORECASE
)

REQUIRED_COLS = {
    "srcs"      : ["Name", "Group"],
    "srcs_locs" : ["Name", "Long", "Lat", "Elevation"],   # ← confirm Lat/Long once srcs_locs loads
    "study"     : ["Name"],
}

# Columns that should always stay as strings
STRING_COLS = {"Group", "Name", "sample", "Sample_ID", "Type", "Unit"}


def _normalise_column_name(raw: str) -> str:
    """
    Clean a single column header:
      1. Strip Bruker spectral-line suffixes  (Ka1, La1, Kb, etc.)
      2. Compress internal whitespace to underscore
      3. Strip leading/trailing whitespace
    
    Examples
    --------
    'Fe Ka1'  → 'Fe'
    'Sr La1'  → 'Sr'
    'Sample Name' → 'Sample_Name'
    '  Zr  '  → 'Zr'
    """
    s = str(raw).strip()
    s = BRUKER_SUFFIX_PATTERN.sub("", s)   # remove Bruker suffix
    s = re.sub(r'\s+', '_', s).strip("_")  # whitespace → underscore
    return s


def clean_geochem_df(
    df        : pd.DataFrame,
    string_cols: set[str] = STRING_COLS,
    *,
    verbose   : bool = True
) -> pd.DataFrame:
    """
    Clean and type-cast a raw geochemical DataFrame.

    Steps
    -----
    1. Normalise column names  (strip Bruker suffixes, compress whitespace)
    2. Promote known string columns to pandas ``StringDtype``
    3. Convert all remaining columns to numeric  (coerce → NaN)
    4. Drop entirely-empty columns
    5. Report what was changed  (if verbose=True)

    Parameters
    ----------
    df          : raw DataFrame straight from load_table()
    string_cols : column names to keep as strings (case-sensitive)
    verbose     : print a cleaning summary

    Returns
    -------
    pd.DataFrame — cleaned copy
    """
    df = df.copy()

    # ── 1. Normalise headers ────────────────────────────────
    old_cols     = list(df.columns)
    df.columns   = [_normalise_column_name(c) for c in old_cols]
    renamed      = {o: n for o, n in zip(old_cols, df.columns) if o != n}

    # ── 2. String columns ───────────────────────────────────
    str_present  = [c for c in string_cols if c in df.columns]
    for col in str_present:
        df[col] = df[col].astype("string")

    # ── 3. Numeric coercion ─────────────────────────────────
    num_cols     = [c for c in df.columns if c not in string_cols]
    before_nulls = df[num_cols].isna().sum().sum()
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
    after_nulls  = df[num_cols].isna().sum().sum()
    new_nulls    = after_nulls - before_nulls

    # ── 4. Drop empty columns ───────────────────────────────
    empty_cols   = df.columns[df.isna().all()].tolist()
    df           = df.dropna(axis=1, how="all")

    # ── 5. Report ───────────────────────────────────────────
    if verbose:
        print("── clean_geochem_df ──────────────────────────")
        print(f"  Shape        : {df.shape[0]} rows × {df.shape[1]} cols")

        if renamed:
            print(f"  Renamed cols ({len(renamed)}):")
            for old, new in renamed.items():
                print(f"    '{old}'  →  '{new}'")
        else:
            print("  Renamed cols : none")

        print(f"  String cols  : {str_present if str_present else 'none'}")
        print(f"  Numeric cols : {len(num_cols)}")
        print(f"  New NaNs     : {new_nulls}  (non-numeric coerced)")
        print(f"  Dropped cols : {empty_cols if empty_cols else 'none'}")
        print("──────────────────────────────────────────────")

    return df


# ── Validation helpers ───────────────────────────────────────

def require_columns(
    df      : pd.DataFrame,
    required: list[str],
    name    : str = "DataFrame"
) -> None:
    """
    Raise a clear ValueError if any required columns are absent.

    Parameters
    ----------
    df       : DataFrame to check
    required : list of column names that must be present
    name     : label used in the error message
    """
    missing = [c for c in required if c not in df.columns]
    if missing:
        close = _suggest_columns(missing, df.columns)
        msg   = (
            f"'{name}' is missing required columns: {missing}\n"
            f"  Available : {sorted(df.columns.tolist())}"
        )
        if close:
            msg += f"\n  Did you mean: {close} ?"
        raise ValueError(msg)


def _suggest_columns(missing: list[str], available) -> list[str]:
    """
    Fuzzy-match missing column names against available ones
    using simple character overlap — no extra packages needed.
    """
    suggestions = []
    for m in missing:
        m_low = m.lower()
        for a in available:
            # flag if the missing name appears anywhere in an available name
            if m_low in a.lower() or a.lower() in m_low:
                suggestions.append(a)
    return list(dict.fromkeys(suggestions))   # deduplicate, preserve order


def validate_ppm_ranges(
    df      : pd.DataFrame,
    lo      : float = 0.0,
    hi      : float = 1_000_000.0,
    *,
    verbose : bool  = True
) -> pd.DataFrame:
    """
    Flag numeric columns containing values outside [lo, hi].
    Returns a summary DataFrame of out-of-range counts.

    Parameters
    ----------
    df      : cleaned geochem DataFrame
    lo / hi : sensible PPM bounds  (default 0 – 1 000 000)
    verbose : print the summary table
    """
    num_df   = df.select_dtypes(include="number")
    out_low  = (num_df < lo).sum()
    out_high = (num_df > hi).sum()

    summary = pd.DataFrame({
        "min"         : num_df.min(),
        "max"         : num_df.max(),
        "n_below_lo"  : out_low,
        "n_above_hi"  : out_high,
        "n_nan"       : num_df.isna().sum(),
    })
    summary = summary[(summary["n_below_lo"] > 0) |
                      (summary["n_above_hi"] > 0) |
                      (summary["n_nan"]      > 0)]

    if verbose:
        if summary.empty:
            print("validate_ppm_ranges: all values within expected bounds ✓")
        else:
            print(f"validate_ppm_ranges: {len(summary)} columns with anomalies ↓")
            display(summary)

    return summary


# ── Quick smoke-test ─────────────────────────────────────────
if __name__ == "__main__":
    _test = pd.DataFrame({
        "Sample Name" : ["A", "B", "C"],
        "Fe Ka1"      : ["12.3", "bad", "15.1"],
        "Sr La1"      : [220, 310, None],
        "Zr"          : [100, 200, 300],
        "Empty Col"   : [None, None, None],
    })
    _clean = clean_geochem_df(_test)
    require_columns(_clean, ["Fe", "Sr", "Zr"], name="test_df")
    print(_clean)
    print(_clean.dtypes)

── clean_geochem_df ──────────────────────────
  Shape        : 3 rows × 3 cols
  Renamed cols (4):
    'Sample Name'  →  'Sample_Name'
    'Fe Ka1'  →  'Fe'
    'Sr La1'  →  'Sr'
    'Empty Col'  →  'Empty_Col'
  String cols  : none
  Numeric cols : 5
  New NaNs     : 4  (non-numeric coerced)
  Dropped cols : ['Sample_Name', 'Empty_Col']
──────────────────────────────────────────────
     Fe     Sr   Zr
0  12.3  220.0  100
1   NaN  310.0  200
2  15.1    NaN  300
Fe    float64
Sr    float64
Zr      int64
dtype: object


In [7]:
# ============================================================
# CELL 4 — Load & clean the three tables
# ============================================================

print("=" * 48)
print(" CELL 4 — Loading & cleaning project tables")
print("=" * 48)

srcs      = clean_geochem_df(load_table("srcs"),      verbose=True)
srcs_locs = clean_geochem_df(load_table("srcs_locs"), verbose=True)
study     = clean_geochem_df(load_table("study"),     verbose=True)

require_columns(srcs,      ["Name", "Group"],               name="srcs")
require_columns(srcs_locs, ["Name", "Long", "Lat"], name="srcs_locs")
require_columns(study,     ["Name"],                        name="study")

# ── Cross-table integrity check ──────────────────────────────
# Every Name in study should exist in srcs
unknown = set(study["Name"].dropna()) - set(srcs["Name"].dropna())
if unknown:
    print(f"\n  ⚠  {len(unknown)} Name(s) in study not found in srcs:")
    for u in sorted(unknown):
        print(f"       '{u}'")
else:
    print("\n  Name cross-check study → srcs ✓")

# Every Name in srcs_locs should exist in srcs
unknown_locs = set(srcs_locs["Name"].dropna()) - set(srcs["Name"].dropna())
if unknown_locs:
    print(f"  ⚠  {len(unknown_locs)} Name(s) in srcs_locs not found in srcs:")
    for u in sorted(unknown_locs):
        print(f"       '{u}'")
else:
    print("  Name cross-check srcs_locs → srcs ✓")

# ── Table summary ────────────────────────────────────────────
print("\n── Table summary ─────────────────────────────")
for label, df in [("srcs", srcs), ("srcs_locs", srcs_locs), ("study", study)]:
    n_num = df.select_dtypes(include="number").shape[1]
    n_str = df.select_dtypes(include="string").shape[1]
    print(f"  {label:<12}  {df.shape[0]:>4} rows  "
          f"{df.shape[1]:>3} cols  "
          f"({n_num} numeric, {n_str} string)")
print("──────────────────────────────────────────────")

for label, df in [("srcs", srcs), ("srcs_locs", srcs_locs), ("study", study)]:
    print(f"\n── {label} preview ──")
    display(df.head(3).style.format(precision=2, na_rep="—"))

 CELL 4 — Loading & cleaning project tables
  → fetching 'KRA21_sources' from Google Sheets...
     loaded  294 rows × 15 cols ✓
── clean_geochem_df ──────────────────────────
  Shape        : 294 rows × 14 cols
  Renamed cols : none
  String cols  : ['Group', 'Name']
  Numeric cols : 13
  New NaNs     : 294  (non-numeric coerced)
  Dropped cols : ['ANID']
──────────────────────────────────────────────
  → fetching 'source_coords' from Google Sheets...
     loaded  7 rows × 3 cols ✓
── clean_geochem_df ──────────────────────────
  Shape        : 7 rows × 3 cols
  Renamed cols : none
  String cols  : ['Name']
  Numeric cols : 2
  New NaNs     : 0  (non-numeric coerced)
  Dropped cols : none
──────────────────────────────────────────────
  → fetching 'study_samples' from Google Sheets...
     loaded  15 rows × 15 cols ✓
── clean_geochem_df ──────────────────────────
  Shape        : 15 rows × 13 cols
  Renamed cols (12):
    'MnKa1'  →  'Mn'
    'ZnKa1'  →  'Zn'
    'GaKa1'  →  'Ga'
    

,Name,Group,Fe,Th,Rb,Sr,Y,Zr,Nb,Northing,Easting,Elevation,Long,Lat
0,Chivay,Chivay,4723,19,241,37,19,86,17,—,—,—,—,—
1,Chivay,Chivay,4701,21,241,39,19,87,17,—,—,—,—,—
2,Chivay,Chivay,4587,20,243,38,18,86,17,—,—,—,—,—



── srcs_locs preview ──


,Name,Lat,Long
0,Alca,-15.16,-72.74
1,Anillo,-15.06,-72.98
2,Chivay,-15.64,-71.54



── study preview ──


,Name,Group,Mn,FeKa1_percent,Zn,Ga,Rb,Sr,Y,Zr,Nb,Ba,Th
0,RGM-Solid1,Reference,196,1.31,32,17,146,104,29,211,8,810,12
1,7150_1,Anillo,408,0.63,44,22,159,142,17,118,16,905,19
2,7150_2,Anillo,405,0.63,47,19,162,146,14,123,15,994,17


In [8]:
# CELL 5 — Prepare interactive map (FigureWidget) and UI widgets
# Ensure latitude/longitude numeric and drop invalid rows
srcs_locs = srcs_locs.dropna(subset=["Lat", "Long"]).copy()
srcs_locs["Lat"] = pd.to_numeric(srcs_locs["Lat"], errors="coerce")
srcs_locs["Long"] = pd.to_numeric(srcs_locs["Long"], errors="coerce")
srcs_locs = srcs_locs.dropna(subset=["Lat", "Long"]).reset_index(drop=True)

names = srcs_locs["Name"].astype(str).tolist()
lats = srcs_locs["Lat"].astype(float).tolist()
lons = srcs_locs["Long"].astype(float).tolist()
center_lat = float(np.mean(lats)) if lats else 0.0
center_lon = float(np.mean(lons)) if lons else 0.0

map_fig = FigureWidget(go.Figure(go.Scattermapbox(
lat=lats, lon=lons, mode="markers+text", text=names, textposition="top center",
marker=dict(size=10, color="royalblue", opacity=0.9), customdata=names, selectedpoints=[]
)))
map_fig.update_layout(
mapbox=dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=9),
dragmode="lasso",
height=480,
margin=dict(r=0, l=0, t=30, b=0),
title="Source locations — use lasso/box to select points"
)

out = widgets.Output(layout={"border": "1px solid lightgray"})
btn_get = widgets.Button(description="Get selection", button_style="primary")
btn_clear = widgets.Button(description="Clear selection", button_style="warning")
btn_save = widgets.Button(description="Save selection CSV", button_style="success", disabled=True)
selection_box = widgets.VBox([widgets.HBox([btn_get, btn_clear, btn_save]), out])

Global variables used by downstream cells
selected_groups = []
selected_srcs_df = pd.DataFrame()
selected_srcs = pd.DataFrame()

SyntaxError: invalid syntax (1789234037.py, line 32)

In [ ]:
print('Complete Study Sample Table')
display(study)

## Use Lasso tool to select obsidian sources from region of interest

In [ ]:
# ============================================================
# CELL 6 — Selection handling functions (clear, capture, save)
# ============================================================

def clear_selection(_=None):
    """Clear map selection and UI state."""
    global selected_groups, selected_srcs_df
    map_fig.data[0].selectedpoints = []
    selected_groups   = []
    selected_srcs_df  = pd.DataFrame()
    btn_save.disabled = True
    with out:
        clear_output()
        print("Selection cleared. Draw a new selection on the map "
              "and click 'Get selection'.")


def _match_groups_by_name(names_list: list) -> pd.DataFrame:
    """
    Find rows in srcs matching any of the provided names.

    Matching strategy
    -----------------
    1. Exact match against ``Group``
    2. Exact match against ``Name``
    3. Case-insensitive containment against ``Group`` / ``Name``  (fallback)

    Returns
    -------
    pd.DataFrame — subset of srcs, may be empty.
    """
    if not names_list:
        return pd.DataFrame()

    df      = srcs.copy()
    matched = pd.DataFrame()

    # Only check columns that actually exist — "Sample" removed
    id_cols = [c for c in ("Group", "Name") if c in df.columns]

    for n in names_list:
        n_str = str(n).strip()
        if not n_str:
            continue

        # ── exact match ──────────────────────────────────────
        cond = pd.Series(False, index=df.index)
        for col in id_cols:
            cond |= df[col].astype(str).str.strip() == n_str

        # ── containment fallback ─────────────────────────────
        if not cond.any():
            for col in id_cols:
                cond |= df[col].astype(str).str.contains(
                    re.escape(n_str), case=False, na=False
                )

        matched = pd.concat([matched, df.loc[cond]])

    if not matched.empty:
        matched = matched.drop_duplicates().reset_index(drop=True)

    return matched


def capture_selection(_=None):
    """
    Read the current lasso/box selection from the map figure,
    resolve selected point names to source rows, and update
    the global ``selected_srcs_df``.
    """
    global selected_groups, selected_srcs_df

    with out:
        clear_output()

        # ── guard: figure must have data ─────────────────────
        if not map_fig.data:
            print("Error: map figure has no data.")
            return

        sel = map_fig.data[0].selectedpoints
        if not sel:
            print("No points selected — draw a lasso/box selection "
                  "then click 'Get selection'.")
            return

        # ── extract names from customdata ────────────────────
        customdata = list(map_fig.data[0].customdata)
        names_sel  = []
        for idx in sel:
            try:
                i = int(idx)
                if 0 <= i < len(customdata):
                    names_sel.append(str(customdata[i]))
            except Exception:
                continue

        # deduplicate, preserve order
        seen      = set()
        names_sel = [x for x in names_sel
                     if not (x in seen or seen.add(x))]

        if not names_sel:
            print("No valid names extracted from selection.")
            return

        # ── match to source rows ─────────────────────────────
        selected_srcs_

In [ ]:
# ============================================================
# CELL 7 — Exploratory XY plot with selectable axes
# ============================================================

import plotly.express as px

# ── Infer plottable elements from loaded tables ───────────────
def _numeric_cols(df: pd.DataFrame) -> list[str]:
    """Return sorted numeric column names, excluding coords/IDs."""
    exclude = {"Easting", "Northing", "Lat", "Long", "Elevation"}
    return sorted([
        c for c in df.select_dtypes(include="number").columns
        if c not in exclude
    ])

_default_cols = _numeric_cols(srcs)   # populate dropdowns from srcs on load

# ── Controls ─────────────────────────────────────────────────
table_choice = widgets.Dropdown(
    options     = ["Selected sources", "Study samples"],
    value       = "Selected sources",
    description = "Table",
    layout      = widgets.Layout(width="220px"),
)
x_choice = widgets.Dropdown(
    options     = _default_cols,
    value       = "Rb" if "Rb" in _default_cols else _default_cols[0],
    description = "X axis",
    layout      = widgets.Layout(width="180px"),
)
y_choice = widgets.Dropdown(
    options     = _default_cols,
    value       = "Sr" if "Sr" in _default_cols else _default_cols[1],
    description = "Y axis",
    layout      = widgets.Layout(width="180px"),
)
log_x = widgets.Checkbox(value=False, description="log X",
                         layout=widgets.Layout(width="80px"))
log_y = widgets.Checkbox(value=False, description="log Y",
                         layout=widgets.Layout(width="80px"))
update_btn = widgets.Button(
    description  = "Update plot",
    button_style = "info",
    icon         = "refresh",
)
viz_out = widgets.Output(
    layout={"border": "1px solid #ddd", "padding": "8px"}
)

# ── Update dropdown options when table changes ────────────────
def _on_table_change(change):
    """Repopulate axis dropdowns when the user switches table."""
    if change["name"] != "value":
        return
    df   = selected_srcs_df if change["new"] == "Selected sources" else study
    cols = _numeric_cols(df)
    x_choice.options = cols
    y_choice.options = cols

table_choice.observe(_on_table_change)

# ── Plot function ─────────────────────────────────────────────
def update_plot(_=None):
    """Build XY scatter from current control values."""
    with viz_out:
        clear_output(wait=True)

        # ── resolve table ─────────────────────────────────────
        if table_choice.value == "Selected sources":
            if selected_srcs_df.empty:
                print("⚠  No sources selected — use the map in Cell 6 first.")
                return
            df = selected_srcs_df
        else:
            df = study

        x_col = x_choice.value
        y_col = y_choice.value

        # ── validate columns ──────────────────────────────────
        for col in (x_col, y_col):
            if col not in df.columns:
                print(f"⚠  Column '{col}' not found in table.")
                return

        # ── drop rows missing either axis value ───────────────
        plot_df = df[[x_col, y_col]
                     + (["Group"] if "Group" in df.columns else [])
                     + (["Name"]  if "Name"  in df.columns else [])
                    ].dropna(subset=[x_col, y_col])

        if plot_df.empty:
            print(f"⚠  No data after dropping NaNs for {x_col} / {y_col}.")
            return

        color_col = "Group" if "Group" in plot_df.columns else None
        hover     = [c for c in ("Name", "Group") if c in plot_df.columns]

        # ── build plot ────────────────────────────────────────
        fig = px.scatter(
            plot_df,
            x          = x_col,
            y          = y_col,
            color      = color_col,
            hover_data = hover,
            log_x      = log_x.value,
            log_y      = log_y.value,
            title      = f"{y_col} vs {x_col}  ({len(plot_df)} samples)",
            template   = "plotly_white",
        )
        fig.update_traces(
            marker=dict(size=9, opacity=0.85,
                        line=dict(width=0.4, color="black"))
        )
        fig.update_layout(
            height      = 520,
            xaxis_title = x_col,
            yaxis_title = y_col,
        )
        display(fig)


# ── Wire & display ────────────────────────────────────────────
update_btn.on_click(update_plot)

display(widgets.VBox([
    widgets.HBox([table_choice, x_choice, y_choice,
                  log_x, log_y, update_btn]),
    viz_out,
]))

# Auto-render on load
update_plot()

In [ ]:
print(selected_locs)

In [ ]:
# CELL 8 — Small helper: quick-export of selected sample names (optional)
def list_selected_sample_names():
"""Return a list of 'Sample' values from the selected sources (if present)."""
if selected_srcs_df.empty:
print("No selected samples.")
return []
if "Sample" in selected_srcs_df.columns:
return selected_srcs_df["Sample"].astype(str).tolist()
return selected_srcs_df.index.astype(str).tolist()CELL 8 — Small helper: quick-export of selected sample names (optional)
def list_selected_sample_names():
"""Return a list of 'Sample' values from the selected sources (if present)."""
if selected_srcs_df.empty:
print("No selected samples.")
return []
if "Sample" in selected_srcs_df.columns:
return selected_srcs_df["Sample"].astype(str).tolist()
return selected_srcs_df.index.astype(str).tolist()

# Example usage (uncomment to run)
# print("Selected sample names:", list_selected_sample_names())

In [ ]:
# CELL 9 — Short notes for commit & run
# - If you plan to commit this notebook to the repo, consider:
# * Adding the exact CSV filenames to the README (KRA21_sources.csv, source_coords.csv, study_samples.csv)
# * Documenting the expected column names: Group, Sample, Name, Lat, Long, plus element columns (Rb, Sr, Zr, ...)
# - To switch to a Google Sheet source: set USE_GOOGLE_SHEETS = True, put the sheet ID in SHEET_ID,
# and ensure the SHEET_TABS keys match the tab names in the sheet.
# - If you want further analysis (ellipses, discriminant analyses, bootstrapping), tell me which tests/plots you want.

In [ ]:
# Ellipses require at least 2 data points per group.
# Split Sources into two tables called 'srcs' and 'onerow' for single sample sources

# Count how many times each group appears
counts = srcs['Group'].value_counts()

# DataFrame with groups that appear < 2 times
onesample = srcs[srcs['Group'].map(counts) < 2]

# DataFrame with sources that have >= 2 samples
srcs = srcs[srcs['Group'].map(counts) >= 2]

print('Ellipses will be created for the following Source Groups:')
print(srcs['Group'].value_counts())

print('Points will be shown for these Sources with <2 samples:')

if onesample is not None and not onesample.empty:
    print(onesample)
else:
    print('(All sources have sufficient samples for ellipses)')


In [ ]:
#@title
#@
def confidence_ellipse(x, y, n_std=1.96, size=100):   # Ellipses in Plotly
    """
    Get the covariance confidence ellipse of *x* and *y*.
    from https://gist.github.com/dpfoose/38ca2f5aee2aea175ecc6e599ca6e973

    Parameters
    ----------
    x, y : array-like, shape (n, )
        Input data.
    n_std : float
        The number of standard deviations to determine the ellipse's radiuses.
    size : int
        Number of points defining the ellipse
    Returns
    -------
    String containing an SVG path for the ellipse

    References (H/T)
    ----------------
    https://matplotlib.org/3.1.1/gallery/statistics/confidence_ellipse.html
    https://community.plotly.com/t/arc-shape-with-path/7205/5
    """
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    pearson = cov[0, 1]/np.sqrt(cov[0, 0] * cov[1, 1])
    # Using a special case to obtain the eigenvalues of this
    # two-dimensionl dataset.
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    theta = np.linspace(0, 2 * np.pi, size)
    ellipse_coords = np.column_stack([ell_radius_x * np.cos(theta), ell_radius_y * np.sin(theta)])

    # Calculating the stdandard deviation of x from
    # the squareroot of the variance and multiplying
    # with the given number of standard deviations.
    x_scale = np.sqrt(cov[0, 0]) * n_std
    x_mean = np.mean(x)

    # calculating the stdandard deviation of y ...
    y_scale = np.sqrt(cov[1, 1]) * n_std
    y_mean = np.mean(y)

    translation_matrix = np.tile([x_mean, y_mean], (ellipse_coords.shape[0], 1))
    rotation_matrix = np.array([[np.cos(np.pi / 4), np.sin(np.pi / 4)],
                                [-np.sin(np.pi / 4), np.cos(np.pi / 4)]])
    scale_matrix = np.array([[x_scale, 0],
                            [0, y_scale]])
    ellipse_coords = ellipse_coords.dot(rotation_matrix).dot(scale_matrix) + translation_matrix

    path = f'M {ellipse_coords[0, 0]}, {ellipse_coords[0, 1]}'
    for k in range(1, len(ellipse_coords)):
        path += f'L{ellipse_coords[k, 0]}, {ellipse_coords[k, 1]}'
    path += ' Z'
    return path



In [ ]:
# Assign a color to each Source for consistency
name_to_color = {}

unique_groups = srcs['Group'].dropna().unique()  # drop NA just in case
colors = DEFAULT_PLOTLY_COLORS

# Cycle colors if more groups than colors
name_to_color = {name: colors[i % len(colors)] for i, name in enumerate(unique_groups)}

# 2️⃣ If you want a simple mapping to original name (optional)
unique_name_to_name = {name: name for name in unique_groups}

In [ ]:
# To change the variables modify x_col and y_col here
# and Re-run the cell
print('Available elements to plot.')
print(srcs.columns)
print('To change elements modify x_col and y_col variables and re-run this cell:')

# Variables to plot
x_col = 'Sr'
y_col = 'Zr'
group = 'Group'

figsrcs = go.Figure()


# Add source ellipses
for g in srcs[group].unique():
    # Map group to color, default to gray if missing
    color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")

    # Optional: add source points (uncomment lines below to show points for sources instead 
    # of just the ellipses)
    # figsrcs.add_trace(
    #     go.Scatter(
    #         x=srcs[srcs[group] == g][x_col],
    #         y=srcs[srcs[group] == g][y_col],
    #         name=g,
    #         mode='markers',
    #         marker=dict(symbol='x', size=4, color=color)
    #     )
    # )

    # Add confidence ellipse for the group
    figsrcs.add_shape(
        type='path',
        path=confidence_ellipse(
            srcs[srcs[group] == g][x_col],
            srcs[srcs[group] == g][y_col]
        ),
        line_color=color,
        name=g,
        showlegend=True
    )

# Add single source sample point
figsrcs.add_trace(
    go.Scatter(
        x=onesample[x_col],
        y=onesample[y_col],
        name="Source Sample",
        mode='markers',
        marker=dict(symbol='x', size=4, color='black'),
        text=onesample['Sample'],
        hovertemplate="Source: %{text}<br><extra></extra>",
        showlegend=True
    )
)

# Map Study sample groups to colors
study_colors = [
    name_to_color.get(unique_name_to_name.get(g, ""), "gray")
    for g in study[group]
]

# Add study samples colored by Group
figsrcs.add_trace(
    go.Scatter(
        x=study[x_col],
        y=study[y_col],
        name="Study Samples",
        mode='markers',
        text=study['Sample'],
        hovertemplate="Sample: %{text}<br><extra></extra>",
        showlegend=True,
        marker=dict(
            size=8,
            symbol='circle',
            color=study_colors
        )
    )
)
# ------------------------------
# Add confidence ellipses for study groups
# ------------------------------
for g in study[group].unique():
    mask = study[group] == g
    data_x = study.loc[mask, x_col]
    data_y = study.loc[mask, y_col]
    
    # Skip empty groups
    if len(data_x) < 2:
        continue
    
    color = name_to_color.get(g, "gray")
    
    figsrcs.add_shape(
        type='path',
        path=confidence_ellipse(data_x, data_y),
        line_color=color,
        line_width=2,
        name=f"{g} Study Ellipse",
        legendgroup=g,
        showlegend=True,
        opacity=0.5
    )

# Set layout with centered title
figsrcs.update_layout(
    title={
        'text': "Connecting Study Samples with known obsidian sources",
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title=x_col,
    yaxis_title=y_col
)

for g in study[group].unique():
    figsrcs.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode="lines",
            line=dict(color=name_to_color.get(g, "gray")),
            name=f"Study {g}",
            showlegend=True
        )
    )

figsrcs.show(renderer="notebook")

In [ ]:
# TERNARY PLOT

# --- Function to compute ellipse in 2D (fractions space) ---
def confidence_ellipse_points(x, y, n_std=1.96, size=100):
    """
    Return Nx2 array of (x,y) points for the confidence ellipse in 2D space.
    """
    if x.size != y.size:
        raise ValueError("x and y must be the same size")

    cov = np.cov(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    eigvals, eigvecs = np.linalg.eigh(cov)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]

    theta = np.linspace(0, 2 * np.pi, size)
    ellipse = np.stack([np.cos(theta), np.sin(theta)], axis=1)
    scale = n_std * np.sqrt(eigvals)
    ellipse = ellipse @ np.diag(scale) @ eigvecs.T
    ellipse += np.array([mean_x, mean_y])

    return ellipse  # Nx2 array

# --- Normalize compositional data ---
def normalize_composition(df, cols):
    """
    Normalize rows so that the specified columns sum to 1
    """
    comp = df[cols].values
    row_sums = comp.sum(axis=1).reshape(-1,1)
    comp_norm = comp / row_sums
    return comp_norm

# Columns
cols = ['Rb','Sr','Zr']

# Normalize srcs and study
srcs_frac = normalize_composition(srcs, cols)
study_frac = normalize_composition(study, cols)

# --- Ternary Plot ---
fig = go.Figure()

# --- PLOT ELLIPSES ---
for g in srcs[group].unique():
    color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")

    mask = srcs[group] == g
    rb, sr, zr = srcs_frac[mask].T

    ell = confidence_ellipse_points(rb, sr)

    eRb = ell[:,0]
    eSr = ell[:,1]
    eZr = 1 - eRb - eSr

    fig.add_trace(
        go.Scatterternary(
            a=eRb, b=eSr, c=eZr,
            mode='lines',
            line=dict(color=color),
            name=f"{g} Source",
            showlegend=True
        )
    )

# --- PLOT ONESAMPLE POINTS IF NON-EMPTY ---
if onesample is not None and not onesample.empty:
    onesample_frac = normalize_composition(onesample, cols)
    for g in onesample[group].unique():
        mask = onesample[group] == g
        data = onesample_frac[mask]
        color = name_to_color.get(unique_name_to_name.get(g, ""), "black")

        fig.add_trace(
            go.Scatterternary(
                a=data[:,0],
                b=data[:,1],
                c=data[:,2],
                mode='markers',
                name=f"Source Sample: {g}",
                marker=dict(symbol='x', size=6, color=color),
                text=onesample[mask]['Sample'],
                hovertemplate="Source: %{text}<extra></extra>"
            )
        )

# --- PLOT STUDY SAMPLES WITH DIFFERENT COLORS PER GROUP ---
for g in study[group].unique():
    mask = study[group] == g
    data = study_frac[mask]
    color = name_to_color.get(unique_name_to_name.get(g, ""), "gray")

    fig.add_trace(
        go.Scatterternary(
            a=data[:,0],
            b=data[:,1],
            c=data[:,2],
            mode='markers',
            name=f"Study: {g}",
            marker=dict(symbol='circle', size=8, color=color),
            text=study[mask]['Sample'],
            hovertemplate="Sample: %{text}<extra></extra>"
        )
    )

# --- Layout ---
fig.update_layout(
    ternary=dict(
        sum=1,
        aaxis=dict(title='Rb'),
        baxis=dict(title='Sr'),
        caxis=dict(title='Zr')
    ),
    title=dict(
        text="Ternary Plot with Obsidian Source Ellipses and Study Samples",
        x=0.5,
        xanchor='center',
        yanchor='top'
    )
)

fig.show()


## The Analysis is complete. You can roll over the 